# Test opendataloader-pdf
Run each cell to inspect the raw ODL output and debug the parser.

In [22]:
import sys, os
sys.path.insert(0, os.path.abspath('src'))
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'config.settings')

PDF_PATH = '/mnt/c/Users/HADDER/Desktop/L3 – COURS DE CALCUL DIFFÉRENTIEL.pdf'  # chemin WSL
CONTENT_TYPE = 'lesson'  # exercise | exam | lesson

print('PDF_PATH:', PDF_PATH)
print('exists:', os.path.exists(PDF_PATH))

PDF_PATH: /mnt/c/Users/HADDER/Desktop/L3 – COURS DE CALCUL DIFFÉRENTIEL.pdf
exists: True


## 1. Raw ODL output (see what the library actually returns)

In [23]:
import tempfile, json
import opendataloader_pdf

with tempfile.TemporaryDirectory() as tmpdir:
    output_dir = os.path.join(tmpdir, 'out')
    os.makedirs(output_dir)
    opendataloader_pdf.convert(
    input_path=[PDF_PATH],
    output_dir=output_dir,
    format='json',
    quiet=False,
    hybrid='docling-fast',
    hybrid_mode='full',   # requis pour les formules
)


    # Find JSON output
    json_path = None
    for root, _, files in os.walk(output_dir):
        for fname in files:
            print('output file:', fname)
            if fname.endswith('.json'):
                json_path = os.path.join(root, fname)

    if json_path:
        with open(json_path, 'r', encoding='utf-8') as f:
            raw = json.load(f)
        # Save for inspection in later cells
        import builtins
        builtins._odl_raw = raw
        print('type:', type(raw))
        if isinstance(raw, dict):
            print('keys:', list(raw.keys()))
        elif isinstance(raw, list):
            print('list length:', len(raw))
    else:
        print('NO JSON FILE FOUND')

Apr 12, 2026 5:35:37 PM org.opendataloader.pdf.processors.DocumentProcessor preprocessing
INFO: File name: /mnt/c/Users/HADDER/Desktop/L3 – COURS DE CALCUL DIFFÉRENTIEL.pdf
Apr 12, 2026 5:35:37 PM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
Apr 12, 2026 5:35:37 PM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
Apr 12, 2026 5:35:37 PM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
Apr 12, 2026 5:35:37 PM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
Apr 12, 2026 5:35:37 PM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
Apr 12, 2026 5:35:37 PM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
Apr 12, 2026 5:35:37 PM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
Apr 12, 2026 5:35:37 PM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
Apr 12, 2026 5:35:37 PM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
Apr 12, 2026 5:35:37 PM org.verapdf.gf.model.factory.chunks.ChunkParser

## 2. Inspect top-level structure

In [16]:
import builtins
raw = builtins._odl_raw

# If it's a dict, show all keys and a sample of each
if isinstance(raw, dict):
    for k, v in raw.items():
        print(f'--- key: {k!r} ---')
        if isinstance(v, list):
            print(f'  list of {len(v)} items')
            if v:
                print('  first item:', json.dumps(v[0], ensure_ascii=False, indent=2)[:500])
        else:
            print(' ', repr(v)[:200])
elif isinstance(raw, list):
    print(f'List of {len(raw)} elements')
    for el in raw[:5]:
        print(json.dumps(el, ensure_ascii=False, indent=2)[:400])
        print('---')

--- key: 'file name' ---
  'M2_cours.pdf'
--- key: 'number of pages' ---
  1
--- key: 'author' ---
  None
--- key: 'title' ---
  None
--- key: 'creation date' ---
  'D:20260412144522'
--- key: 'modification date' ---
  None
--- key: 'kids' ---
  list of 14 items
  first item: {
  "type": "heading",
  "id": 15,
  "level": "Doctitle",
  "page number": 1,
  "bounding box": [
    86.174,
    691.956,
    188.598,
    713.655
  ],
  "heading level": 1,
  "font": null,
  "font size": 12.0,
  "content": "Chapter 1"
}


## 3. Extract element list and show all element types present

In [17]:
raw = builtins._odl_raw

# Try all known locations for the elements list
if isinstance(raw, list):
    elements = raw
elif isinstance(raw, dict):
    elements = (
        raw.get('elements') or
        raw.get('content') or
        raw.get('blocks') or
        raw.get('pages') or
        []
    )
    # If pages, flatten
    if elements and isinstance(elements[0], dict) and 'elements' in elements[0]:
        flat = []
        for page in elements:
            flat.extend(page.get('elements', []))
        elements = flat

builtins._elements = elements
print(f'Total elements: {len(elements)}')

from collections import Counter
type_counts = Counter(el.get('type', 'UNKNOWN') for el in elements)
print('Element types:', dict(type_counts))

Total elements: 0
Element types: {}


## 4. Show all elements with their text content

In [18]:
elements = builtins._elements

for i, el in enumerate(elements[:50]):  # first 50
    el_type = el.get('type', '?')
    content = el.get('content', el.get('text', el.get('value', '')))
    if isinstance(content, list):
        content_str = str(content)[:120]
    else:
        content_str = str(content)[:120]
    level = el.get('level', '')
    level_str = f' [level={level}]' if level else ''
    print(f'[{i:03d}] {el_type}{level_str}: {content_str!r}')

## 5. Show full JSON of a specific element (change index)

In [19]:
elements = builtins._elements
INDEX = 0  # <-- change this
print(json.dumps(elements[INDEX], ensure_ascii=False, indent=2))

IndexError: list index out of range

## 6. Test current parser end-to-end

In [20]:
from apps.things.pdf_parser import parse_pdf

with open(PDF_PATH, 'rb') as f:
    pdf_bytes = f.read()

result = parse_pdf(pdf_bytes, CONTENT_TYPE)
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "title": "Sans titre",
  "sections": []
}


## 7. Check raw output as markdown (easier to read)

In [26]:
# Cellule de test — markdown avec hybrid activé (serveur doit tourner)
import tempfile, os
import opendataloader_pdf

with tempfile.TemporaryDirectory() as tmpdir:
    output_dir = os.path.join(tmpdir, 'out')
    os.makedirs(output_dir)
    opendataloader_pdf.convert(
        input_path=[PDF_PATH],
        output_dir=output_dir,
        format='markdown',
        quiet=False,
        hybrid='docling-fast',
        hybrid_mode='full',   # active formula enrichment
    )
    for root, _, files in os.walk(output_dir):
        for fname in files:
            fpath = os.path.join(root, fname)
            print(f'=== {fname} ===')
            with open(fpath, 'r', encoding='utf-8') as f:
                print(f.read()[:5000])


Apr 12, 2026 5:40:24 PM org.opendataloader.pdf.processors.DocumentProcessor preprocessing
INFO: File name: /mnt/c/Users/HADDER/Desktop/L3 – COURS DE CALCUL DIFFÉRENTIEL.pdf
Apr 12, 2026 5:40:24 PM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
Apr 12, 2026 5:40:24 PM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
Apr 12, 2026 5:40:24 PM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
Apr 12, 2026 5:40:24 PM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
Apr 12, 2026 5:40:24 PM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
Apr 12, 2026 5:40:24 PM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
Apr 12, 2026 5:40:24 PM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
Apr 12, 2026 5:40:24 PM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
Apr 12, 2026 5:40:24 PM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
Apr 12, 2026 5:40:24 PM org.verapdf.gf.model.factory.chunks.ChunkParser